# Pulse at a string junction

*waves · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/string_junction)

Two strings with different mass densities are tied together. A pulse coming from the left splits at the knot into a reflected and a transmitted pulse whose amplitudes are set by the impedances $Z = \sqrt{T\mu}$. The second derivative is a 1D convolution with the stencil $(1, -2, 1)$.

**Vector form**

$$\mu(x)\,\frac{\partial^2 u}{\partial t^2} = T\,\frac{\partial^2 u}{\partial x^2},\qquad R = \frac{Z_1 - Z_2}{Z_1 + Z_2},\qquad \mathcal T = \frac{2Z_1}{Z_1 + Z_2}$$

**Index (tensor) form**

$$\mu_i\,\ddot u_i = T\,D_{ij}\,u_j,\qquad D_{ij}u_j = \frac{u_{i+1} - 2u_i + u_{i-1}}{\Delta x^2},\qquad R^2 + \frac{Z_2}{Z_1}\,\mathcal T^2 = 1$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_i$ | `u` | (N,) | displacement at grid point $i$; both far ends are fixed |
| $\mu_i$ | `density(x, mu1, mu2)` | (N,) | mass per length: $\mu_1$ left, $\mu_2$ right, the average on the knot |
| $D_{ij}u_j$ | `conv1d(u, [1, -2, 1]) / dx**2` | (N,) | repeated $j$ ⇒ summed: the stencil slides along the string |
| $Z_k = \sqrt{T\mu_k}$ | `impedance(T, mu)` | scalar | wave impedance of string $k$ |
| $R,\ \mathcal T$ | `reflection_transmission(Z1, Z2)` | scalar | amplitude ratios from continuity of $u$ and of the force $T\,u_x$ at the knot |
| $E$ | `energy(u_prev, u, u_next, mu, T, dx, dt)` | scalar | $\sum_i \tfrac12\mu_i\dot u_i^2\Delta x + \tfrac12 T (\partial_x u)^2\Delta x$, conserved |

The knot cannot break, so $u$ is continuous, and it is massless, so the transverse forces $T u_x$ on both sides balance. These two conditions give $R$ and $\mathcal T$. A heavier right string ($Z_2 > Z_1$) reflects the pulse upside down, like a fixed end; a lighter one reflects it upright, like a free end. The transmitted pulse is always upright and moves at $c_2 = \sqrt{T/\mu_2}$, so it is squeezed or stretched by $c_2/c_1$. The energy carried by a pulse is proportional to $Z\,A^2$, which gives the flux balance $R^2 + (Z_2/Z_1)\,\mathcal T^2 = 1$. Units: $T = 1$, $\mu_1 = 1$, string from $-1$ to $1$. Grey dashed lines mark the predicted amplitudes. Watch the faster string: its pulse reaches the far wall and bounces back.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/string_junction/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    T, mu1, mu2 = tensors.TENSION, tensors.MU1, params["mu_ratio"] * tensors.MU1
    c1, c2 = tensors.speed(mu1), tensors.speed(mu2)
    x, dx = tensors.grid(params["n_points"], dev)
    mu = tensors.density(x, mu1, mu2)
    dt = 0.9 * dx / max(c1, c2)  # Courant condition on the faster string
    u_prev, u0 = tensors.right_moving_pulse(x, params["width"], c1, dt)

    t_meas = solve.measuring_time(c1, c2, tensors.X0)
    times, traj = solve.solve(u_prev, u0, mu, T, dx, dt, max(params["t_max"], t_meas))

    Z1, Z2 = equation.impedance(T, mu1), equation.impedance(T, mu2)
    R, Tr = equation.reflection_transmission(Z1, Z2)
    R_num, T_num = solve.measure(x, traj[(times - t_meas).abs().argmin()])
    print(f"Z1 = {Z1:.3f}, Z2 = {Z2:.3f},  c1 = {c1:.3f} m/s, c2 = {c2:.3f} m/s")
    print(f"reflected   amplitude {R_num:+.4f}   theory R = (Z1-Z2)/(Z1+Z2) = {R:+.4f}")
    print(f"transmitted amplitude {T_num:+.4f}   theory T = 2Z1/(Z1+Z2)     = {Tr:+.4f}")
    print(f"energy flux balance R² + (Z2/Z1) T² = {R_num**2 + Z2 / Z1 * T_num**2:.4f}  (should be 1)")
    E = equation.energy(traj, mu, T, dx, dt)
    print(f"largest relative energy change: {((E - E[0]).abs().max() / E[0]).item():.2e}")

    shown = times <= params["t_max"] + 1e-9
    shown[0] = False  # skip the t = −Δt row
    info = {"R": R, "T": Tr, "Z1": Z1, "Z2": Z2, "mu2": mu2}
    return plot.animate(times[shown], x, traj[shown], info, out_dir / "string_junction.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/string_junction/equation.py
r"""Physics → tensor translation.

A string under tension T whose mass per length μ(x) jumps at the knot x = 0:

    \mu(x) \, \partial_t^2 u = T \, \partial_x^2 u

On the grid the second derivative is a stencil applied at every point,

    \mu_i \ddot u_i = T D_{ij} u_j,        D_{ij} u_j = (u_{i+1} - 2 u_i + u_{i-1}) / \Delta x^2    (j summed)

which is exactly a 1D convolution with the kernel (1, −2, 1). Matching u and T u_x at the knot gives

    R = (Z_1 - Z_2) / (Z_1 + Z_2),     \mathcal T = 2 Z_1 / (Z_1 + Z_2),     Z = \sqrt{T \mu}.
"""

import math

import torch
import torch.nn.functional as F


def second_derivative(u: torch.Tensor, dx: float) -> torch.Tensor:
    r"""D_{ij} u_j by convolution with (1, −2, 1) / Δx². u: (N,) → (N,), zero at the two fixed ends."""
    kernel = torch.tensor([1.0, -2.0, 1.0], dtype=u.dtype, device=u.device).view(1, 1, 3)
    inner = F.conv1d(u.view(1, 1, -1), kernel).view(-1) / dx**2  # (N − 2,) interior points only
    return F.pad(inner, (1, 1))  # the fixed ends do not move


def acceleration(u: torch.Tensor, mu: torch.Tensor, T: float, dx: float) -> torch.Tensor:
    r"""\ddot u_i = (T / μ_i) D_{ij} u_j.   (N,) → (N,)"""
    return T / mu * second_derivative(u, dx)


def impedance(T: float, mu: float) -> float:
    """Z = √(T μ)."""
    return math.sqrt(T * mu)


def reflection_transmission(Z1: float, Z2: float) -> tuple[float, float]:
    """Amplitude ratios R = (Z1 − Z2)/(Z1 + Z2) and 𝒯 = 2 Z1/(Z1 + Z2)."""
    return (Z1 - Z2) / (Z1 + Z2), 2 * Z1 / (Z1 + Z2)


def energy(traj: torch.Tensor, mu: torch.Tensor, T: float, dx: float, dt: float) -> torch.Tensor:
    r"""E = Σ_i ½ μ_i \dot u_i² Δx + ½ T Σ (Δu/Δx)² Δx at every time.

    traj: (S, N). The velocity is the centred difference (u^{n+1} − u^{n−1}) / 2Δt, so the result is (S − 2,).
    """
    v = (traj[2:] - traj[:-2]) / (2 * dt)  # (S-2, N)
    ux = (traj[1:-1, 1:] - traj[1:-1, :-1]) / dx  # (S-2, N-1)
    return 0.5 * torch.einsum("i,si->s", mu, v**2) * dx + 0.5 * T * torch.einsum("si,si->s", ux, ux) * dx


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/string_junction/tensors.py
"""Build the concrete tensors: the grid, the density profile and the incoming pulse."""

import math

import torch

TENSION = 1.0
MU1 = 1.0
X0 = -0.5  # starting position of the pulse


def grid(n_points: int, device: torch.device) -> tuple[torch.Tensor, float]:
    """x of shape (N,) from −1 to 1 (fixed ends) and the spacing Δx."""
    return torch.linspace(-1.0, 1.0, n_points, device=device), 2.0 / (n_points - 1)


def density(x: torch.Tensor, mu1: float, mu2: float) -> torch.Tensor:
    """μ_i of shape (N,): μ1 for x < 0, μ2 for x > 0 and the average on the knot."""
    mu = torch.where(x < 0, mu1, mu2)
    return torch.where(x.abs() < 1e-9, 0.5 * (mu1 + mu2), mu)


def right_moving_pulse(x: torch.Tensor, width: float, c1: float, dt: float) -> tuple[torch.Tensor, torch.Tensor]:
    """A Gaussian u(x, t) = exp(−(x − x0 − c1 t)² / 2σ²) at t = 0 and t = −Δt: two (N,) tensors.

    Giving the leap-frog both time levels launches the pulse to the right only.
    """

    def gauss(t: float) -> torch.Tensor:
        return torch.exp(-((x - X0 - c1 * t) ** 2) / (2 * width**2))

    return gauss(-dt), gauss(0.0)


def speed(mu: float) -> float:
    """c = √(T / μ)."""
    return math.sqrt(TENSION / mu)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/string_junction/solve.py
"""March the string with leap-frog and read off the reflected and transmitted amplitudes."""

import torch

from problems._common.progress import steps

from .equation import acceleration


def solve(
    u_prev: torch.Tensor, u: torch.Tensor, mu: torch.Tensor, T: float, dx: float, dt: float, t_max: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (S,), trajectory (S, N)). Row 0 is t = −Δt, row 1 is t = 0."""
    n_steps = int(round(t_max / dt))
    frames = [u_prev, u]
    for _ in steps(n_steps, "time steps"):
        u, u_prev = 2 * u - u_prev + dt**2 * acceleration(u, mu, T, dx), u
        frames.append(u)
    times = torch.arange(-1, n_steps + 1, device=u.device) * dt
    return times, torch.stack(frames)


def signed_peak(u: torch.Tensor) -> float:
    """The value of largest magnitude, with its sign (a pulse may be upside down)."""
    return u[u.abs().argmax()].item()


def measure(x: torch.Tensor, u: torch.Tensor) -> tuple[float, float]:
    """Reflected (x < 0) and transmitted (x > 0) pulse amplitudes in one snapshot u (N,)."""
    return signed_peak(u[x < 0]), signed_peak(u[x > 0])


def measuring_time(c1: float, c2: float, x0: float) -> float:
    """A moment when both pulses have cleared the knot but the transmitted one has not reached the far wall."""
    t_cross = abs(x0) / c1
    return t_cross + min(0.4, 0.8 / c2)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/string_junction/plot.py
"""Animate the string: the heavier/lighter half is shaded, dashed lines show the predicted R and 𝒯."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(times: torch.Tensor, x: torch.Tensor, traj: torch.Tensor, info: dict, path: Path) -> Path:
    t, xs, u = times.cpu().numpy(), x.cpu().numpy(), traj.cpu().numpy()  # (S,), (N,), (S, N)
    R, Tr = info["R"], info["T"]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.axvspan(0, 1, color="#93c5fd", alpha=0.25, lw=0)
    ax.axvline(0, color="#374151", lw=1)
    ax.plot([-1, 0], [R, R], "--", color="#9ca3af", lw=1)
    ax.plot([0, 1], [Tr, Tr], "--", color="#9ca3af", lw=1)
    ax.text(-0.98, R, f" R = {R:+.3f}", va="bottom" if R >= 0 else "top", fontsize=9, color="#4b5563")
    ax.text(0.98, Tr, rf"$\mathcal{{T}}$ = {Tr:.3f} ", ha="right", va="bottom", fontsize=9, color="#4b5563")
    ax.text(-0.5, 1.3, f"μ₁ = 1,  Z₁ = {info['Z1']:.2f}", ha="center", fontsize=9)
    ax.text(0.5, 1.3, f"μ₂ = {info['mu2']:.2g},  Z₂ = {info['Z2']:.2f}", ha="center", fontsize=9)
    line = ax.plot(xs, u[0], color="#2563eb", lw=2)[0]
    ax.plot([-1, 1], [0, 0], "o", color="#374151", ms=5)
    ax.set_xlim(-1.02, 1.02)
    ax.set_ylim(-1.1, 1.5)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("u")
    title = ax.set_title("string displacement   t = 0.00 s")

    def update(step: int):
        line.set_ydata(u[step])
        title.set_text(f"string displacement   t = {t[step]:.2f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'mu_ratio': 4.0,
    'width': 0.04,
    't_max': 1.2,
    'n_points': 1001
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.string_junction")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.string_junction.main")
problem_dir = Path("problems/waves/string_junction")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
